# 00 · Pipeline completo

**Objetivo.** Ejecutar todo el flujo desde un único Jupyter: **Miner (opcional) →
Analyzer → Visualizer**, y mostrar el tablero inline. Es el *entry point* del
proyecto; los notebooks `01`-`04` son exploración y validación **opcional**.

**Entrada.**
- `results.json` (scan) si existe; si no, los que existan de `results-sbom.json`
  y `results-vuln.json` (en ese orden).
- Variables de entorno: `MINER_MODE` (`auto`/`force`/`off`), `MINER_ORGANIZATION`,
  `MINER_LIMIT` y `GENERATED_AT`.

**Salida.**
- `analysis/outputs/analyzer_output.json` (contrato validado, `schema_version` 1.1).
- `analysis/outputs/visualizer.html` (tablero autocontenido).

**Supuestos.**
- `auto` reutiliza reportes si existen; si no, ejecuta el Miner. `force` siempre
  ejecuta y `off` nunca.
- El Miner solo se ejecuta si hace falta y con `MINER_ORGANIZATION` y
  `GITHUB_TOKEN` presentes; el token nunca se imprime.
- Todo es determinista salvo por `meta.generated_at`, que se fija con
  `GENERATED_AT` (p. ej. `GENERATED_AT="2026-10-05T12:00:00+00:00"`).
- Es tolerante: si ya existe `analyzer_output.json` y no hay reportes, en modo
  `auto` reutiliza el documento previo sin invocar al Miner.


## 0. Parámetros y rutas

Se resuelve la raíz del repo (donde está `pyproject.toml`) y se leen las
variables de entorno. Todas las rutas se derivan de `REPO_ROOT`.


In [ ]:
from pathlib import Path

import os


def _find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file():
            return candidate
    return start


REPO_ROOT = _find_repo_root(Path.cwd())

MODE = (os.environ.get("MINER_MODE") or "auto").strip() or "auto"
ORGANIZATION = (os.environ.get("MINER_ORGANIZATION") or "").strip() or None
GENERATED_AT = os.environ.get("GENERATED_AT")

_raw_limit = (os.environ.get("MINER_LIMIT") or "").strip()
if _raw_limit:
    try:
        LIMIT = int(_raw_limit)
    except ValueError as exc:
        raise ValueError(f"MINER_LIMIT debe ser un entero: {_raw_limit!r}") from exc
else:
    LIMIT = None

SCAN_REPORT = REPO_ROOT / "results.json"
ANALYZER_OUTPUT = REPO_ROOT / "analysis" / "outputs" / "analyzer_output.json"
OUTPUT_HTML = REPO_ROOT / "analysis" / "outputs" / "visualizer.html"
WORKDIR = REPO_ROOT / "workdir"
SBOM_DIR = REPO_ROOT / "sboms"
VULN_DIR = REPO_ROOT / "vulns"

print("REPO_ROOT          :", REPO_ROOT)
print("MINER_MODE         :", MODE)
print("MINER_ORGANIZATION :", ORGANIZATION or "(no definida)")
print("MINER_LIMIT        :", LIMIT)
print("GENERATED_AT       :", GENERATED_AT)
print("ANALYZER_OUTPUT    :", ANALYZER_OUTPUT)
print("OUTPUT_HTML        :", OUTPUT_HTML)


## 1. Preparar el entorno

Se inserta la raíz del repo en `sys.path` y se importan la orquestación del
Miner, el pipeline del Analyzer, el Visualizer y el visor inline de IPython.


In [ ]:
import sys

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from IPython.display import IFrame

from analysis.orchestrator import (
    build_miner_command,
    existing_reports,
    github_token_present,
    run_command,
    should_run_miner,
)
from analysis.pipeline import run_analysis
from miner.visualizer import build_visualizer, load_document


## 2. Paso 1 · Miner (opcional)

`should_run_miner(REPO_ROOT, MODE)` decide si hay que ejecutar el Miner. Si hay
que hacerlo, se exige `MINER_ORGANIZATION`, se comprueba que `GITHUB_TOKEN` esté
definido (sin imprimir su valor) y se lanza `miner scan` con los directorios de
trabajo del repo.

**Tolerancia.** En modo `auto`, si no hay reportes pero ya existe
`analyzer_output.json`, se omite el Miner y se reutiliza el documento previo.


In [ ]:
INPUT_PATHS = existing_reports(REPO_ROOT)
RUN_MINER = should_run_miner(REPO_ROOT, MODE)

# Tolerancia: no hay reportes, pero sí un documento del Analyzer previo.
if RUN_MINER and MODE == "auto" and not INPUT_PATHS and ANALYZER_OUTPUT.is_file():
    RUN_MINER = False
    print("Sin reportes del Miner; se reutiliza el documento del Analyzer existente.")

if RUN_MINER:
    if not ORGANIZATION:
        raise ValueError(
            "MINER_ORGANIZATION es obligatoria para ejecutar el Miner "
            f"(MINER_MODE={MODE!r})."
        )
    if not github_token_present():
        raise RuntimeError(
            "GITHUB_TOKEN no está definido; el Miner lo necesita para la API de "
            "GitHub."
        )

    command = build_miner_command(
        organization=ORGANIZATION,
        output=SCAN_REPORT,
        limit=LIMIT,
        repos_dir=WORKDIR,
        sbom_dir=SBOM_DIR,
        vuln_dir=VULN_DIR,
    )
    print("Ejecutando Miner:", " ".join(command))
    exit_code = run_command(command, cwd=REPO_ROOT, echo=print)
    if exit_code != 0:
        raise RuntimeError(f"El Miner terminó con código {exit_code}.")
else:
    print("Miner omitido; se reutilizan los reportes existentes.")

INPUT_PATHS = existing_reports(REPO_ROOT)

if not INPUT_PATHS and not ANALYZER_OUTPUT.is_file():
    raise FileNotFoundError(
        "No se encontró results.json, results-sbom.json ni results-vuln.json"
    )

print("INPUT_PATHS:")
for path in INPUT_PATHS:
    print("  -", path)


## 3. Paso 2 · Analyzer

`run_analysis` fusiona los reportes por repositorio, calcula métricas y escribe
el contrato validado. Si no hay reportes pero existe un documento previo, se
reutiliza tal cual (tolerancia).


In [ ]:
if INPUT_PATHS:
    document = run_analysis(INPUT_PATHS, ANALYZER_OUTPUT, generated_at=GENERATED_AT)
    ORIGEN = "run_analysis"
else:
    document = load_document(ANALYZER_OUTPUT)
    ORIGEN = "documento del Analyzer existente"

meta = document["meta"]
datasets = document["datasets"]

print("Origen           :", ORIGEN)
print("Organización     :", meta["organization"])
print("Repositorios     :", meta["repositories"])
print("Vulnerabilidades :", len(datasets["vulnerabilities"]))
print("Nota global      :", f"{datasets['risk_summary']['score']}/10")
print("source_kind      :", meta["source_kind"])


## 4. Paso 3 · Visualizer

`build_visualizer` embebe el documento y los assets en un único HTML. Se valida
que exista, que no esté vacío, que incluya el bloque de datos del Analyzer y que
no referencie recursos externos de terceros (excluyendo las URLs del propio JSON
de datos).


In [ ]:
TITLE = f"Visualizer · {meta['organization']}"
built = build_visualizer(ANALYZER_OUTPUT, OUTPUT_HTML, title=TITLE)

html = OUTPUT_HTML.read_text(encoding="utf-8")

assert OUTPUT_HTML.is_file(), OUTPUT_HTML
assert html.strip(), "El HTML generado está vacío"
assert '<script id="analyzer-data" type="application/json">' in html, (
    "Falta el bloque de datos del Analyzer"
)

# Se recorta el JSON embebido: sus URLs de datos son legítimas.
DATA_MARKER = '<script id="analyzer-data" type="application/json">'
data_start = html.index(DATA_MARKER) + len(DATA_MARKER)
data_end = html.index("</script>", data_start)
html_without_data = html[:data_start] + html[data_end:]

EXTERNAL_MARKERS = ('src="http', 'href="http', '@import url(http')
external = [marker for marker in EXTERNAL_MARKERS if marker in html_without_data]
assert not external, f"Recursos externos detectados: {external}"

print("HTML                        :", built)
print("Bloque de datos del Analyzer : OK")
print("Sin recursos externos        : OK")


## 5. Paso 4 · Vista inline

Se muestra el tablero generado dentro del notebook mediante un `IFrame` con la
ruta relativa al directorio de trabajo.


In [ ]:
try:
    shown = OUTPUT_HTML.relative_to(Path.cwd())
except ValueError:
    shown = OUTPUT_HTML

IFrame(src=str(shown), width="100%", height=900)


## 6. Resumen final

Verificación de los artefactos declarados y de sus rutas y tamaños.


In [ ]:
assert ANALYZER_OUTPUT.is_file() and ANALYZER_OUTPUT.stat().st_size > 0, (
    f"Artefacto ausente o vacío: {ANALYZER_OUTPUT}"
)
assert OUTPUT_HTML.is_file() and OUTPUT_HTML.stat().st_size > 0, (
    f"Artefacto ausente o vacío: {OUTPUT_HTML}"
)


def _relative(path: Path) -> Path:
    try:
        return path.relative_to(REPO_ROOT)
    except ValueError:
        return path


print("Pipeline completo")
print("-----------------")
print("Organización     :", meta["organization"])
print("Repositorios     :", meta["repositories"])
print("Vulnerabilidades :", len(datasets["vulnerabilities"]))
print("Nota global      :", f"{datasets['risk_summary']['score']}/10")
print(
    "Analyzer         :",
    _relative(ANALYZER_OUTPUT),
    f"({ANALYZER_OUTPUT.stat().st_size} bytes)",
)
print(
    "Visualizer       :",
    _relative(OUTPUT_HTML),
    f"({OUTPUT_HTML.stat().st_size} bytes)",
)
